# AI6130 Assignment 2 - FIXED VERSION
## Fixes for AdapterP Bug, LoRA Config Missing, and P100 Compatibility

**This notebook fixes 3 critical bugs:**
1. ✅ AdapterP training (`config` variable bug)
2. ✅ LoRA model validation (missing config.json)
3. ✅ PyTorch P100 GPU compatibility

**Run this single cell on Kaggle with P100 GPU**

In [ ]:
# ============================================================================
# AI6130 ASSIGNMENT 2 - FIXED COMPLETE PIPELINE
# ============================================================================
# Fixes:
# 1. PyTorch reinstall for P100 compatibility (sm_60)
# 2. finetune.py patching for AdapterP bug
# 3. Model validation before evaluation
# ============================================================================

import os
import subprocess
import sys
import time
import pandas as pd
import re
from pathlib import Path

print("="*80)
print("AI6130 ASSIGNMENT 2 - FIXED PIPELINE")
print("="*80)

# ============================================================================
# STEP 1: FIX PYTORCH FOR P100 GPU (sm_60 compatibility)
# ============================================================================
print("\n[STEP 1/9] Fixing PyTorch for P100 GPU...")

# Uninstall current PyTorch
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'], 
               capture_output=True)

# Install PyTorch with CUDA 11.8 (supports sm_60 for P100)
print("Installing PyTorch with CUDA 11.8 (P100 compatible)...")
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.0.1+cu118',
    'torchvision==0.15.2+cu118', 
    'torchaudio==2.0.2+cu118',
    '--index-url', 'https://download.pytorch.org/whl/cu118'
], check=True)
print("✓ PyTorch reinstalled for P100")

# Verify GPU detection
import torch
if torch.cuda.is_available():
    print(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"✓ CUDA version: {torch.version.cuda}")
else:
    print("⚠ WARNING: No GPU detected!")

# ============================================================================
# STEP 2: SETUP ENVIRONMENT
# ============================================================================
print("\n[STEP 2/9] Setting up environment...")

os.environ["WANDB_MODE"] = "disabled"
WORK_DIR = '/kaggle/working/AI6130_Assignment2'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"✓ Working directory: {WORK_DIR}")

# ============================================================================
# STEP 3: CLONE REPOSITORY
# ============================================================================
print("\n[STEP 3/9] Cloning repository...")

REPO_DIR = os.path.join(WORK_DIR, 'AI6130_Assignment2')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/duyngtr16061999/AI6130_Assignment2'], check=True)
    print("✓ Repository cloned")
else:
    print("✓ Repository exists")

os.chdir(REPO_DIR)

# ============================================================================
# STEP 4: INSTALL DEPENDENCIES
# ============================================================================
print("\n[STEP 4/9] Installing dependencies...")

packages = ['fire', 'datasets', 'transformers==4.38.0', 'accelerate==0.28.0']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'peft', '-y'], capture_output=True)
print("✓ Dependencies installed")

# ============================================================================
# STEP 5: FIX FINETUNE.PY FOR ADAPTERP BUG
# ============================================================================
print("\n[STEP 5/9] Patching finetune.py for AdapterP bug...")

finetune_path = 'finetune.py'

# Read the file
with open(finetune_path, 'r') as f:
    content = f.read()

# Check if already patched
if 'PATCHED_FOR_ADAPTERP' not in content:
    # Find the problematic section (around line 223)
    # The bug is: config variable is referenced before being assigned
    # We need to ensure config is initialized in all code paths
    
    # Add patch marker
    patch = '''\n    # PATCHED_FOR_ADAPTERP - Fix UnboundLocalError
    config = None  # Initialize config to avoid UnboundLocalError
'''
    
    # Insert before the line that causes the error
    # Look for: model = get_peft_model(model, config)
    content = content.replace(
        'model = get_peft_model(model, config)',
        patch + '    if config is None:\n' +
        '        raise ValueError(f"Unknown adapter: {adapter_name}. Supported: lora, AdapterP")\n' +
        '    model = get_peft_model(model, config)'
    )
    
    # Write patched version
    with open(finetune_path, 'w') as f:
        f.write(content)
    
    print("✓ finetune.py patched successfully")
else:
    print("✓ finetune.py already patched")

# ============================================================================
# STEP 6: DELETE INCOMPLETE MODELS
# ============================================================================
print("\n[STEP 6/9] Cleaning up incomplete models...")

def is_model_valid(model_dir):
    """Check if model has all required files"""
    if not os.path.exists(model_dir):
        return False
    
    required_files = ['adapter_config.json', 'adapter_model.bin']
    for file in required_files:
        if not os.path.exists(os.path.join(model_dir, file)):
            return False
    return True

# Check existing models
trained_models_dir = './trained_models'
if os.path.exists(trained_models_dir):
    for model_name in os.listdir(trained_models_dir):
        model_path = os.path.join(trained_models_dir, model_name)
        if os.path.isdir(model_path):
            if not is_model_valid(model_path):
                print(f"⚠ Removing incomplete model: {model_name}")
                import shutil
                shutil.rmtree(model_path)
            else:
                print(f"✓ Valid model found: {model_name}")

# ============================================================================
# STEP 7: FINE-TUNE ALL MODELS
# ============================================================================
print("\n[STEP 7/9] Fine-tuning all models...")

EXPERIMENTS = [
    {'name': 'OpenLLaMA-LoRA', 'base': 'openlm-research/open_llama_3b_v2', 'adapter': 'lora', 'out': './trained_models/openlm-lora'},
    {'name': 'OpenLLaMA-AdapterP', 'base': 'openlm-research/open_llama_3b_v2', 'adapter': 'AdapterP', 'out': './trained_models/openlm-adapterp'},
    {'name': 'TinyLlama-LoRA', 'base': 'TinyLlama/TinyLlama_v1.1', 'adapter': 'lora', 'out': './trained_models/tinyllama-lora'},
    {'name': 'TinyLlama-AdapterP', 'base': 'TinyLlama/TinyLlama_v1.1', 'adapter': 'AdapterP', 'out': './trained_models/tinyllama-adapterp'},
]

training_results = []

for i, exp in enumerate(EXPERIMENTS, 1):
    print(f"\n{'='*80}")
    print(f"Training {i}/{len(EXPERIMENTS)}: {exp['name']}")
    print(f"{'='*80}")
    
    if is_model_valid(exp['out']):
        print(f"✓ Model already trained, skipping")
        training_results.append({'Model': exp['name'], 'Status': 'Skipped', 'Time': 'N/A'})
        continue
    
    cmd = [
        'python', 'finetune.py',
        '--base_model', exp['base'],
        '--data_path', './ft-training_set/math_7k.json',
        '--output_dir', exp['out'],
        '--batch_size', '4',
        '--micro_batch_size', '4',
        '--num_epochs', '1',
        '--learning_rate', '3e-4',
        '--cutoff_len', '256',
        '--val_set_size', '120',
        '--adapter_name', exp['adapter']
    ]
    
    start = time.time()
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True, timeout=7200)
        elapsed = time.time() - start
        
        # Verify model was saved properly
        if is_model_valid(exp['out']):
            print(f"\n✓ Training completed: {elapsed/60:.1f} min")
            training_results.append({'Model': exp['name'], 'Status': 'Success', 'Time': f"{elapsed/60:.1f} min"})
        else:
            print(f"\n✗ Training failed: model files incomplete")
            training_results.append({'Model': exp['name'], 'Status': 'Failed (incomplete)', 'Time': f"{elapsed/60:.1f} min"})
            
    except subprocess.TimeoutExpired:
        print(f"\n✗ Training timeout (>2 hours)")
        training_results.append({'Model': exp['name'], 'Status': 'Timeout', 'Time': '>120 min'})
    except subprocess.CalledProcessError as e:
        elapsed = time.time() - start
        print(f"\n✗ Training failed: {e}")
        print(f"\nError output (last 500 chars):\n{e.stderr[-500:]}")
        training_results.append({'Model': exp['name'], 'Status': 'Failed', 'Time': f"{elapsed/60:.1f} min"})

print("\n" + "="*80)
print("Training Summary:")
print(pd.DataFrame(training_results).to_string(index=False))

# ============================================================================
# STEP 8: EVALUATE ALL MODELS
# ============================================================================
print("\n[STEP 8/9] Evaluating all models...")

DATASETS = ['AQuA', 'AddSub']
evaluation_results = []

for exp in EXPERIMENTS:
    # Only evaluate if model is valid
    if not is_model_valid(exp['out']):
        print(f"\n⚠ Skipping {exp['name']}: model not trained")
        for ds in DATASETS:
            evaluation_results.append({
                'Model': exp['name'],
                'Dataset': ds,
                'Accuracy': 'N/A',
                'Status': 'No model'
            })
        continue
    
    for ds in DATASETS:
        print(f"\n{'='*80}")
        print(f"Evaluating: {exp['name']} on {ds}")
        print(f"{'='*80}")
        
        # Determine adapter name for evaluate.py
        adapter_arg = 'LoRA' if exp['adapter'] == 'lora' else exp['adapter']
        
        cmd = [
            'python', 'evaluate.py',
            '--adapter', adapter_arg,
            '--dataset', ds,
            '--base_model', exp['base'],
            '--lora_weights', exp['out']
        ]
        
        start = time.time()
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True, timeout=1800)
            elapsed = time.time() - start
            
            # Parse accuracy
            output = result.stdout
            print(output[-800:])
            
            acc_match = re.search(r'[Aa]cc(?:uracy)?[:\s]+(\d+\.?\d*)%?', output)
            accuracy = acc_match.group(1) if acc_match else 'Check output'
            
            print(f"\n✓ Evaluation done: {elapsed:.1f}s, Accuracy: {accuracy}%")
            evaluation_results.append({
                'Model': exp['name'],
                'Dataset': ds,
                'Accuracy': f"{accuracy}%",
                'Status': 'Success'
            })
            
        except subprocess.TimeoutExpired:
            print(f"\n✗ Evaluation timeout")
            evaluation_results.append({
                'Model': exp['name'],
                'Dataset': ds,
                'Accuracy': 'Timeout',
                'Status': 'Timeout'
            })
        except subprocess.CalledProcessError as e:
            print(f"\n✗ Evaluation failed: {e}")
            print(f"\nError (last 500 chars):\n{e.stderr[-500:]}")
            evaluation_results.append({
                'Model': exp['name'],
                'Dataset': ds,
                'Accuracy': 'Failed',
                'Status': 'Failed'
            })

# ============================================================================
# STEP 9: GENERATE FINAL RESULTS
# ============================================================================
print("\n[STEP 9/9] Generating final results...")

results_df = pd.DataFrame(evaluation_results)

if not results_df.empty:
    # Pivot table
    pivot = results_df.pivot_table(
        index='Model',
        columns='Dataset',
        values='Accuracy',
        aggfunc='first'
    )
    
    print("\n" + "="*80)
    print("📊 FINAL RESULTS TABLE (COPY TO REPORT)")
    print("="*80)
    print(pivot.to_string())
    
    print("\n\n📋 DETAILED RESULTS")
    print("="*80)
    print(results_df[['Model', 'Dataset', 'Accuracy']].to_string(index=False))
    
    # Save CSV files
    results_df.to_csv('results_detailed.csv', index=False)
    pivot.to_csv('results_pivot.csv')
    print("\n✓ Results saved: results_detailed.csv, results_pivot.csv")

# Summary
print("\n" + "="*80)
print("✅ PIPELINE COMPLETED")
print("="*80)
successful = len([r for r in evaluation_results if r['Status'] == 'Success'])
print(f"\n📊 Statistics:")
print(f"  • Successful evaluations: {successful}/{len(EXPERIMENTS)*len(DATASETS)}")
print(f"  • Models trained: {len([r for r in training_results if r['Status'] == 'Success'])}")
print(f"\n📁 Output files in: {os.getcwd()}")
print("\n" + "="*80)